# Neural approximate best response

This notebook calibrates neural Double-DQN best responders against exact best response on `r4_s4_h2_hp2pt_ss`. It compares ordinary sampled-action rollouts with one-step expansion of every legal responder action. Both variants receive the same measured training-time budget; held-out evaluations are excluded from that budget.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.serialization import load_policy, save_policy
from liars_poker.policies.tabular_dense import DenseTabularPolicy
from liars_poker.algo.br_exact_dense_to_dense import best_response_dense
from liars_poker.algo.br_neural import NeuralBRTrainer
from liars_poker.training.br_neural import neural_br_timed_loop
from liars_poker.eval.approx_br import evaluate_approximate_br

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__)
print('device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

# This existing CFR+ run gives us a meaningful frozen target immediately.
# Replace this path with another saved DenseTabularPolicy to test a different target.
target_dir = (
    REPO_ROOT / 'artifacts' / 'benchmark_runs' / 'cfr_plus_runs'
    / 'r4_s4_h2_hp2pt_ss___20260108-213016' / 'policy'
)
target_policy, target_spec = load_policy(target_dir)
assert target_spec == spec
assert isinstance(target_policy, DenseTabularPolicy)
print(target_policy)

## Exact response ceiling

`p_first` and `p_second` are the exact win probabilities of an optimal responder when it plays first and second respectively. The approximate responder's role-specific win rates should approach these values from below.

In [ ]:
_, exact_meta = best_response_dense(
    spec,
    target_policy,
    debug=True,
    store_state_values=False,
)
exact_first, exact_second = exact_meta['computer'].exploitability()
exact_exploitability = exact_first + exact_second - 1.0
pd.Series({
    'exact BR win rate as first': exact_first,
    'exact BR win rate as second': exact_second,
    'exact exploitability': exact_exploitability,
})

## Equal-time sampled versus expanded responders

`sampled` stores only the action used by the live trajectory. `all_actions` stores a one-step transition for every legal responder action, while continuing the episode down only the epsilon-greedy selected branch.

In [ ]:
TRAINING_SECONDS = 5 * 60
EVAL_EVERY_SECONDS = 60
EPISODES_PER_ROLE = 4096
ROLLOUT_BATCH_SIZE = 4096
EVAL_EPISODES_PER_ROLE = 100_000

trainer_kwargs = dict(
    hidden_sizes=(512, 512),
    device=device,
    replay_capacity=1_000_000,
    batch_size=4096,
    learning_rate=1e-3,
    train_steps=100,
    warmup_transitions=20_000,
    target_update_every=500,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_decisions=500_000,
)

def run_variant(expansion, seed):
    trainer = NeuralBRTrainer(
        spec,
        target_policy,
        expansion=expansion,
        seed=seed,
        **trainer_kwargs,
    )
    policy, logs, trainer = neural_br_timed_loop(
        trainer,
        TRAINING_SECONDS,
        episodes_per_role=EPISODES_PER_ROLE,
        rollout_batch_size=ROLLOUT_BATCH_SIZE,
        eval_every_seconds=EVAL_EVERY_SECONDS,
        eval_episodes_per_role=EVAL_EPISODES_PER_ROLE,
        debug=True,
    )
    return {'policy': policy, 'logs': logs, 'trainer': trainer}

runs = {
    'sampled action': run_variant('sampled', seed=7),
    'all actions, one step': run_variant('all_actions', seed=7),
}

In [ ]:
rows = []
heldout = {}
for name, run in runs.items():
    result = evaluate_approximate_br(
        run['trainer'],
        episodes_per_role=200_000,
        rollout_batch_size=8192,
    )
    heldout[name] = result
    training = run['logs']['training_series']
    last = training[-1]
    role0, role1 = last['roles']
    rows.append({
        'variant': name,
        'iterations': last['iter'],
        'p_first': result['p_first'],
        'exact p_first gap': exact_first - result['p_first'],
        'p_second': result['p_second'],
        'exact p_second gap': exact_second - result['p_second'],
        'exploitability estimate': result['exploitability_estimate'],
        'conservative lower bound': result['exploitability_lower_bound'],
        'exact exploitability gap': exact_exploitability - result['exploitability_estimate'],
        'role 1 replay seen': role0['replay_seen'],
        'role 2 replay seen': role1['replay_seen'],
        'final role 1 loss': role0['loss'],
        'final role 2 loss': role1['loss'],
    })

summary = pd.DataFrame(rows).set_index('variant')
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for name, run in runs.items():
    series = run['logs']['evaluation_series']
    x = [point['measured_training_s'] / 60 for point in series]
    axes[0].plot(x, [point['p_first'] for point in series], marker='o', label=name)
    axes[1].plot(x, [point['p_second'] for point in series], marker='o', label=name)
    axes[2].plot(
        x,
        [point['exploitability_lower_estimate'] for point in series],
        marker='o',
        label=name,
    )

axes[0].axhline(exact_first, color='black', linestyle='--', label='exact BR')
axes[1].axhline(exact_second, color='black', linestyle='--', label='exact BR')
axes[2].axhline(exact_exploitability, color='black', linestyle='--', label='exact')
axes[0].set_title('Responder plays first')
axes[1].set_title('Responder plays second')
axes[2].set_title('Discovered exploitability')
for ax in axes:
    ax.set_xlabel('Measured training minutes')
    ax.set_ylabel('Win rate' if ax is not axes[2] else 'Exploitability')
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()

In [ ]:
# Collection and fitting cost at the end of each run.
timing_rows = []
for name, run in runs.items():
    recent = run['logs']['training_series'][-10:]
    for role in (0, 1):
        timing_rows.append({
            'variant': name,
            'role': role + 1,
            'mean collect s': sum(r['roles'][role]['collect_s'] for r in recent) / len(recent),
            'mean fit s': sum(r['roles'][role]['fit_s'] for r in recent) / len(recent),
            'mean transitions': sum(r['roles'][role]['transitions'] for r in recent) / len(recent),
            'final epsilon': recent[-1]['roles'][role]['epsilon'],
        })
pd.DataFrame(timing_rows).set_index(['variant', 'role'])

## Exact-resume and playable-policy storage

The trainer checkpoint contains Q networks, target networks, optimizer state, replay buffers, counters and RNG state. The playable greedy response is separately saved through the repository policy serializer.

In [ ]:
checkpoint_root = REPO_ROOT / 'artifacts' / 'neural_br_demo'
checkpoint_root.mkdir(parents=True, exist_ok=True)

chosen = runs['all actions, one step']
checkpoint_path = checkpoint_root / 'trainer.pt'
policy_dir = checkpoint_root / 'policy'
chosen['trainer'].save_checkpoint(checkpoint_path)
save_policy(chosen['policy'], policy_dir)

resumed = NeuralBRTrainer.load_checkpoint(
    checkpoint_path,
    target_policy,
    device=device,
)
loaded_policy, loaded_spec = load_policy(policy_dir)
print('resumed iteration:', resumed.iteration)
print('replay sizes:', [buffer.size for buffer in resumed.replay])
print('loaded policy:', loaded_policy)
assert loaded_spec == spec